# Assignment 2B — Retrieval-Augmented Generation (RAG) Pipeline
**Group No. 7**  
**Course:** LLM4GenAI  
**Domain:** Financial Annual Reports (Apple, Amazon, NVIDIA, Tesla, Berkshire Hathaway)

---

## Pipeline Overview
```
Domain .txt Corpus
       ↓
  Part A: Chunking (Fixed-Size / Sliding Window / Semantic)
       ↓
  Part B: Retrieval (Dense FAISS / Sparse BM25 / Hybrid RRF)
       ↓
  Part C1: Cross-Encoder Reranking
  Part C2: Tabular RAG (PDF tables → serialised rows → indexed)
```

---
## 📦 Step 1.1 — Install Dependencies

Install all required libraries. Run this cell first.
- `sentence-transformers` — for dense embeddings
- `faiss-cpu` — vector index for dense retrieval
- `rank_bm25` — BM25 sparse retrieval
- `pdfplumber` — extract tables from PDFs
- `transformers` — cross-encoder reranking model
- `nltk` — sentence tokenisation for semantic chunking

In [ ]:
import sys

# Install all required packages
!{sys.executable} -m pip install sentence-transformers faiss-cpu rank_bm25 pdfplumber \
    pandas numpy transformers nltk tqdm --quiet

print("✅ All dependencies installed successfully.")

---
## 📂 Step 1.2 — Load Corpus

Unzip the domain corpus from Assignment 1A. It contains 5 cleaned financial annual report
text files: Apple, Amazon, NVIDIA, Tesla, and Berkshire Hathaway.
We load each file separately so we can track which company each chunk comes from.

In [ ]:
import zipfile
import os
import pandas as pd

# Extract corpus zip into a local folder
CORPUS_ZIP = "ASSIGNMENT1stuff/domain_corpus (2).zip"
CORPUS_DIR = "corpus"

os.makedirs(CORPUS_DIR, exist_ok=True)

with zipfile.ZipFile(CORPUS_ZIP, 'r') as z:
    z.extractall(CORPUS_DIR)

# Load each .txt file and record word count
corpus_files = {}
stats = []

for fname in sorted(os.listdir(CORPUS_DIR)):
    if fname.endswith('.txt'):
        fpath = os.path.join(CORPUS_DIR, fname)
        with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
        company = fname.replace('.txt', '')
        word_count = len(text.split())
        corpus_files[company] = text
        stats.append({'Company': company, 'File': fname, 'Words': word_count, 'Chars': len(text)})
        print(f"  Loaded {company:<15} → {word_count:>7,} words")

# Combine all documents into one corpus string
full_corpus = "\n\n".join(corpus_files.values())
total_words = sum(s['Words'] for s in stats)

print(f"\n{'='*45}")
print(f"  Total documents : {len(corpus_files)}")
print(f"  Total words     : {total_words:,}")
print(f"  Total chars     : {len(full_corpus):,}")
print(f"{'='*45}")

df_corpus_stats = pd.DataFrame(stats)
display(df_corpus_stats)

---
## 🌐 Step 1.3 — Download Annual Report PDFs (for Tabular RAG in Part C)

We re-download the original 5 financial annual report PDFs from Assignment 1B.
These are needed for Part C2 (Tabular RAG) where we extract tables using pdfplumber.
Financial PDFs are rich in structured tables: income statements, balance sheets, segment data.

**Note:** If any download fails (network/size issues), we fall back to using alternate
publicly available financial PDFs — the assignment explicitly permits this.

In [ ]:
import urllib.request
import time

PDF_DIR = "domain_pdfs"
os.makedirs(PDF_DIR, exist_ok=True)

# Original PDF URLs from Assignment 1B
PDF_URLS = {
    'apple':      'https://d18rn0p25nwr6d.cloudfront.net/CIK-0000320193/b4266e40-1de6-4a34-9dfb-8632b8bd57e0.pdf',
    'amazon':     'https://s2.q4cdn.com/299287126/files/doc_financials/2024/ar/Amazon-com-Inc-2023-Annual-Report.pdf',
    'nvidia':     'https://s201.q4cdn.com/141608511/files/doc_financials/2024/ar/NVIDIA-2024-Annual-Report.pdf',
    'tesla':      'https://digitalassets.tesla.com/tesla-contents/image/upload/IR/TSLA-Q4-2023-Update.pdf',
    'berkshire':  'https://www.berkshirehathaway.com/2023ar/2023ar.pdf',
}

# SEC EDGAR requires a User-Agent header to avoid 403 errors
HEADERS = {
    'User-Agent': 'Assignment2B/1.0 2024ad05187@wilp.bits-pilani.ac.in',
    'Accept': 'application/pdf,*/*'
}

downloaded_pdfs = {}
failed_pdfs = []

for company, url in PDF_URLS.items():
    out_path = os.path.join(PDF_DIR, f"{company}.pdf")
    
    # Skip if already downloaded
    if os.path.exists(out_path) and os.path.getsize(out_path) > 10_000:
        size_mb = os.path.getsize(out_path) / 1e6
        print(f"  ✅ {company:<12} already exists ({size_mb:.1f} MB)")
        downloaded_pdfs[company] = out_path
        continue
    
    try:
        print(f"  ⬇️  Downloading {company}...", end=' ', flush=True)
        req = urllib.request.Request(url, headers=HEADERS)
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = resp.read()
        
        # Verify it's actually a PDF
        if not data.startswith(b'%PDF'):
            raise ValueError("Response is not a valid PDF")
        
        with open(out_path, 'wb') as f:
            f.write(data)
        
        size_mb = len(data) / 1e6
        print(f"✅ ({size_mb:.1f} MB)")
        downloaded_pdfs[company] = out_path
        time.sleep(1)  # be polite to servers
        
    except Exception as e:
        print(f"❌ FAILED: {e}")
        failed_pdfs.append(company)

print(f"\n  Downloaded: {len(downloaded_pdfs)}/5 PDFs")
if failed_pdfs:
    print(f"  Failed: {failed_pdfs} — will use fallback PDFs for tabular extraction")

---
## ✅ Step 1.4 — Corpus Summary

Confirm corpus is loaded and ready. Log total word count and per-document breakdown.
This corpus will be used for all chunking and retrieval experiments in Parts A and B.

In [ ]:
# Final corpus summary before chunking
print("CORPUS READY FOR CHUNKING")
print("=" * 45)
for s in stats:
    bar = '█' * (s['Words'] // 5000)
    print(f"  {s['Company']:<12} {s['Words']:>8,} words  {bar}")
print("-" * 45)
print(f"  {'TOTAL':<12} {total_words:>8,} words")
print(f"\n  Estimated chunks @ 200 words (fixed-size): ~{total_words // 200:,}")
print(f"  Estimated chunks @ sliding window (+10%):  ~{int(total_words / 180):,}")
print("=" * 45)